In [1]:
import os
if os.getcwd().endswith('scripts'):
    os.chdir('..')
print(f"Current working directory: {os.getcwd()}")

Current working directory: /data6/yuzihan/WorkSpace/35-HuaweiCrowdSimulation/HuaweiCrowdSimulationCode


In [2]:
from src.dataset import ETHDataset, UCYDataset, SDDDataset, GCDataset, WayMoDataset
from argparse import Namespace
from src.utils.logger import init_logger

init_logger('src')

args = Namespace(
    name="train",
    exp_name="20251009_train_122049_LM2",
    device="cuda:0",
    batch_size=256,
    lr=0.001,
    epochs=10000,
    patience=20,
    sampling_method="DDIM",
    T=100,
    sample_num=1,
    denoise_step=5,
    hist_step=8,
    pred_step=1,
    skip_step=1,
    roll_step=12,
    fps=2.5,
    dot_per_meter=5,
    seed=550,
    save_dir="./logs/train",
    debug=False,
    model_dim=128,
    map_feature_dim=64,
    head_num=4,
    dropout=0.3,
    latent_token_num=16,
    beta_schedule="cosine",
    num_workers=0,
    datasets="ETH/UCY",
    test_name=None,
    test_ratio=None,
    split_by_scenario=False,
    cache_dataset=True,
    test_before_train=True,
    test_per_epoch=10,
    save_per_epoch=50,
    reload_checkpoint=None,
    save_path="logs/train/20251009_train_122049_LM2",
)

In [5]:
import numpy as np
import pandas as pd
from tqdm import tqdm

dataset_list = [
    *ETHDataset.load_data_batch(args, "./data/ETH/"),
    *UCYDataset.load_data_batch(args, "./data/UCY/"),
    GCDataset.load_data(args, "./data/GC/Annotation"),
    *SDDDataset.load_data_batch(args, "./data/SDD/"),
]

df_list = []
for dataset in tqdm(dataset_list):
    df_data = dataset.df_data
    map_data = dataset.map_data
    df_data = df_data[df_data['type'] == 'pedestrian']
    traj_length = np.mean([traj[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).sum() for id, traj in df_data.groupby('id')])
    traj_duration = np.mean([(traj['f'].max() - traj['f'].min()) / args.fps for id, traj in df_data.groupby('id')])

    row = {
        'dataset': type(dataset).__name__.removesuffix('Dataset'),
        '#Trajectory': df_data['id'].nunique(),
        '#Frame': df_data['f'].nunique(),
        'Time': (df_data['f'].max() - df_data['f'].min()) / args.fps,
        'Area': (map_data.xmax - map_data.xmin) * (map_data.ymax - map_data.ymin),
        'Speed': df_data[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).mean() * args.fps,
        'Trajectory-Length': traj_length,
        'Trajectory-Duration': traj_duration,
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

df.groupby('dataset').agg(**{
    '#Scenario': ('#Trajectory', 'count'),
    '#Pedestrian': ('#Trajectory', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time': ('Time', 'sum'),
    'Average Area': ('Area', 'mean'),
    'Average Trajectory-Length': ('Trajectory-Length', 'mean'),
    'Average Trajectory-Duration': ('Trajectory-Duration', 'mean'),
    'Average Speed': ('Speed', 'mean'),
    # 'Area': ('Area', 'sum'),
    # 'Trajectory-Length': ('Trajectory-Length', 'sum'),
    # 'Trajectory-Duration': ('Trajectory-Duration', 'sum'),
    # 'Average Time': ('Time', 'mean'),
})

[None|eth_dataset|I|Oct23 22:00:19|0:07:04.402998] Loading cached dataset-list from data/.cache/ETH.pkl
Loading ETH datasets: 100%|██████████| 2/2 [00:00<00:00, 51.79it/s, seq_hotel]
[None|ucy_dataset|I|Oct23 22:00:20|0:07:04.445030] Loading cached dataset-list from data/.cache/UCY.pkl
Loading UCY datasets: 100%|██████████| 7/7 [00:00<00:00, 66.39it/s, data_zara/crowds_zara03]
[None|gc_dataset|I|Oct23 22:00:20|0:07:04.553883] Loading cached dataset from data/.cache/GC_2.5_8_1_1.pkl
[None|sdd_dataset|I|Oct23 22:00:20|0:07:05.337840] Loading cached dataset-list from data/.cache/SDD.pkl
100%|██████████| 70/70 [00:20<00:00,  3.45it/s]


,#Scenario,#Pedestrian,#Frame,Time,Average Area,Average Trajectory-Length,Average Trajectory-Duration,Average Speed
dataset,,,,,,,,
ETH,2,749,2016,1186.4,490.838498,9.455978,5.785079,2.864477
GC,1,12675,12000,4799.6,3706.106396,33.767133,38.765349,1.587395
SDD,60,13912,42946,17364.0,5673.564034,25.884626,13.928104,3.619905
UCY,7,1480,4460,1801.2,341.527181,14.898826,16.124060,1.722657


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

args.cache_dataset = False
dataset_list = [
    # *ETHDataset.load_data_batch(args, "./data/ETH/"),
    # *UCYDataset.load_data_batch(args, "./data/UCY/"),
    # GCDataset.load_data(args, "./data/GC/Annotation"),
    # *SDDDataset.load_data_batch(args, "./data/SDD/"),
    *WayMoDataset.load_data_batch(args, "./data/WayMo/Processed"),
]

df_list = []
for dataset in tqdm(dataset_list):
    df_data = dataset.df_data
    map_data = dataset.map_data
    df_data = df_data[df_data['type'] == 'pedestrian']
    traj_length = np.mean([traj[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).sum() for id, traj in df_data.groupby('id')])
    traj_duration = np.mean([(traj['f'].max() - traj['f'].min()) / args.fps for id, traj in df_data.groupby('id')])

    row = {
        'dataset': type(dataset).__name__.removesuffix('Dataset'),
        '#Trajectory': df_data['id'].nunique(),
        '#Frame': df_data['f'].nunique(),
        'Time': (df_data['f'].max() - df_data['f'].min()) / args.fps,
        'Area': (map_data.xmax - map_data.xmin) * (map_data.ymax - map_data.ymin),
        'Speed': df_data[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).mean() * args.fps,
        'Trajectory-Length': traj_length,
        'Trajectory-Duration': traj_duration,
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

df.groupby('dataset').agg(**{
    '#Scenario': ('#Trajectory', 'count'),
    '#Pedestrian': ('#Trajectory', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time': ('Time', 'sum'),
    'Average Area': ('Area', 'mean'),
    'Average Trajectory-Length': ('Trajectory-Length', 'mean'),
    'Average Trajectory-Duration': ('Trajectory-Duration', 'mean'),
    'Average Speed': ('Speed', 'mean'),
    # 'Area': ('Area', 'sum'),
    # 'Trajectory-Length': ('Trajectory-Length', 'sum'),
    # 'Trajectory-Duration': ('Trajectory-Duration', 'sum'),
    # 'Average Time': ('Time', 'mean'),
})

[None|waymo_dataset|I|Oct09 14:39:58|0:00:22.874223] Caching dataset-list to data/.cache/WayMo-Processed.pkl
Loading SDD datasets:   0%|          | 0/1788 [00:00<?, ?it/s, Processed/00001_2aa43fad083efbf3][None|base_dataset|I|Oct09 14:39:58|0:00:23.361409] Normalized x with mean=125.4211, std=1.0000
[None|base_dataset|I|Oct09 14:39:58|0:00:23.362507] Normalized y with mean=700.9705, std=1.0000
[None|base_dataset|I|Oct09 14:39:58|0:00:23.364026] Normalized map into xmin=-91.1004, xmax=-48.1998, ymin=75.5691, ymax=104.2940
100%|██████████| 30/30 [00:00<00:00, 33.32it/s]
[None|waymo_dataset|I|Oct09 14:39:59|0:00:24.272919] Caching dataset to data/.cache/00001_2aa43fad083efbf3_2.5_8_1_1.pkl
Loading SDD datasets:   0%|          | 1/1788 [00:01<41:32,  1.39s/it, Processed/00004_e2030d66ebfe7b6b][None|base_dataset|I|Oct09 14:39:59|0:00:24.649467] Normalized x with mean=-831.6093, std=1.0000
[None|base_dataset|I|Oct09 14:39:59|0:00:24.651490] Normalized y with mean=-697.3996, std=1.0000
[None|

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

dataset_list = [
    *ETHDataset.load_data_batch(args, "./data/ETH/"),
    *UCYDataset.load_data_batch(args, "./data/UCY/"),
    GCDataset.load_data(args, "./data/GC/Annotation"),
    *SDDDataset.load_data_batch(args, "./data/SDD/"),
    *WayMoDataset.load_data_batch(args, "./data/WayMo/Processed", total=200),
]

df_list = []
for dataset in tqdm(dataset_list):
    df_data = dataset.df_data
    map_data = dataset.map_data
    traj_length_list = []
    traj_duration_list = []
    traj_speed_list = []
    for id, group in df_data.sort_values(['id', 'f']).groupby('id'):
        if group['type'].nunique() != 1:
            raise ValueError(f"Mixed types in trajectory id {id}")
        if group.iloc[0]['type'] != 'pedestrian':
            continue
        ds = group[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5)
        dt = group['f'].diff() / args.fps
        traj_length_list.append(ds.sum())
        traj_duration_list.append(dt.sum())
        traj_speed_list.append(ds.sum() / dt.sum() if dt.sum() > 0 else 0.0)
    traj_length = np.mean(traj_length_list)
    traj_duration = np.mean(traj_duration_list)
    traj_speed = np.mean(np.array(traj_speed_list)[np.isfinite(traj_speed_list)])

    row = {
        'dataset': type(dataset).__name__.removesuffix('Dataset'),
        'name': dataset.name,
        '#Pedestrian': df_data[df_data['type'] == 'pedestrian']['id'].nunique(),
        '#Vehicle': df_data[df_data['type'] == 'vehicle']['id'].nunique(),
        '#Frame': df_data['f'].nunique(),
        'Time': (df_data['f'].max() - df_data['f'].min()) / args.fps,
        'Area': (map_data.xmax - map_data.xmin) * (map_data.ymax - map_data.ymin),
        'Average-Speed': traj_speed,
        'Average-Length': traj_length,
        'Average-Duration': traj_duration,
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

df.groupby('dataset').agg(**{
    '#Scenario': ('dataset', 'count'),
    '#Pedestrian': ('#Pedestrian', 'sum'),
    '#Vehicle': ('#Vehicle', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time (s)': ('Time', 'sum'),
    # 'Area (m²)': ('Area', 'sum'),
    'Avg Area (m²)': ('Area', 'mean'),
    'Avg Length (m)': ('Average-Length', 'mean'),
    'Avg Duration (s)': ('Average-Duration', 'mean'),
    'Avg Speed (m/s)': ('Average-Speed', 'mean'),
})

[None|eth_dataset|I|Oct23 22:35:03|0:41:48.232698] Loading cached dataset-list from data/.cache/ETH.pkl
Loading ETH datasets: 100%|██████████| 2/2 [00:00<00:00, 45.37it/s, seq_hotel]
[None|ucy_dataset|I|Oct23 22:35:03|0:41:48.280620] Loading cached dataset-list from data/.cache/UCY.pkl
Loading UCY datasets: 100%|██████████| 7/7 [00:00<00:00, 70.82it/s, data_zara/crowds_zara03]
[None|gc_dataset|I|Oct23 22:35:03|0:41:48.383028] Loading cached dataset from data/.cache/GC_2.5_8_1_1.pkl
[None|sdd_dataset|I|Oct23 22:35:04|0:41:49.033625] Loading cached dataset-list from data/.cache/SDD.pkl
Loading SDD datasets: 100%|██████████| 60/60 [00:01<00:00, 45.21it/s, quad/video3]
[None|waymo_dataset|I|Oct23 22:35:05|0:41:50.364490] Loading cached dataset-list from data/.cache/WayMo-Processed.pkl
Loading WayMo datasets:   0%|          | 0/22740 [00:00<?, ?it/s, Processed/00000_12_9859cd1b4315b7de][None|waymo_dataset|I|Oct23 22:35:06|0:41:50.426111] Loading cached dataset from data/.cache/00000_12_9859

,#Scenario,#Pedestrian,#Vehicle,#Frame,Time,Avg Area (m²),Avg Length (m),Avg Duration (s),Avg Speed (m/s)
dataset,,,,,,,,,
ETH,2,749,0,2016,1186.4,490.838498,9.455978,5.785079,1.761384
GC,1,12675,0,12000,4799.6,3706.106396,33.767133,38.765349,1.245894
SDD,60,13912,5067,43189,17364.4,5673.564034,25.884626,13.928104,2.566208
UCY,7,1480,0,4460,1801.2,341.527181,14.898826,16.124060,1.121061
WayMo,200,2564,1222,9542,3771.6,35854.746073,10.727917,8.800703,1.306734


In [27]:
df_SDD = df[df['dataset'] == 'SDD'].copy()
df_SDD['name-1'] = df_SDD['name'].apply(lambda x: x.split('-')[0])
df_SDD = df_SDD.groupby('name-1').agg(**{
    '#Scenario': ('dataset', 'count'),
    '#Pedestrian': ('#Pedestrian', 'sum'),
    '#Vehicle': ('#Vehicle', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time (s)': ('Time', 'sum'),
    'Avg Area (m²)': ('Area', 'mean'),
    'Avg Length (m)': ('Average-Length', 'mean'),
    'Avg Duration (s)': ('Average-Duration', 'mean'),
    'Avg Speed (m/s)': ('Average-Speed', 'mean'),
})
df_SDD

,#Scenario,#Pedestrian,#Vehicle,#Frame,Time (s),Avg Area (m²),Avg Length (m),Avg Duration (s),Avg Speed (m/s)
name-1,,,,,,,,,
bookstore,7,1857,749,8349,3352.8,2392.926031,22.991514,15.624469,1.866880
coupa,4,480,128,3968,1593.6,1870.831973,23.700563,17.338615,1.745713
deathCircle,5,3921,1913,3344,1335.6,13095.852319,36.938709,10.303382,4.511969
gates,9,1521,738,3911,1560.8,6167.158759,30.642001,13.164299,2.874898
hyang,15,3280,747,10549,4302.4,5735.834415,23.272798,13.995412,2.080584
little,4,857,429,3642,1455.2,3789.044074,32.219972,12.603456,3.481596
nexus,12,1956,355,9258,3698.4,7344.025793,24.549772,15.854989,1.886401
quad,4,40,8,168,65.6,1468.584873,16.073503,9.090000,4.428903
